# Exp 028 — top-3 candidates in LM prompt (coupling-fix test)

**Single-axis change vs 021 champion**: pass the wRRF top-3 tracks to the LM prompt (as three numbered 'Candidate' entries) instead of just the top-1.

**Why**: our 6 prior experiments all tanked LLM judge when retrieval changed, because the LM prompt receives ONE recommended track (`recommend_item`) and the LM confidently describes it. Whenever a reranker moved a wrong track to top-1, the LM hallucinated around it and Gemini penalized. Passing top-3 lets the LM pick the best match itself → decouples retrieval top-1 quality from LM-cited-track quality.

**Test conditions**:
- **nDCG@20 should stay ~0.19** (retrieval stack unchanged — wRRF, no reranker).
- **LLM judge is the signal**. If 3.15+ → hypothesis validated, retrieval experiments can resume with other changes. If materially lower (~2.7) → LM can't handle 3-candidate format in the stock prompt and we need targeted prompt tweak.

Wall time: ~4 min on A100 (same as 021 — no extra compute).

Output → `/content/prediction.zip` + Drive backup.

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) Clone fresh-model.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%ndate:    %ai%nsubject: %s'
print()

In [ ]:
# 3) Install deps.
!pip install -q -r requirements.txt
!python -c "import torch, transformers, bm25s; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
# 4) Experiment parameters.
TID = '028-top3-wrrf-qwen15b-blindsetA'
BATCH_SIZE = 32
ATTN = 'sdpa'
# import os; os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 5) Run Blind-A inference with top_n_for_prompt=3.
!cd music-crs-baselines && PYTORCH_ALLOC_CONF=expandable_segments:True \
    python run_inference_blindset.py \
    --tid {TID} \
    --eval_dataset blindset_A \
    --batch_size {BATCH_SIZE} \
    --device cuda \
    --attn_implementation {ATTN}

In [ ]:
# 6) Validate + package prediction.zip.
import json, os, shutil
SRC = f'music-crs-baselines/exp/inference/blindset_A/{TID}.json'
assert os.path.isfile(SRC)
with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)}')
assert len(rows) == 80
sample = rows[0]
required = {'session_id','user_id','turn_number','predicted_track_ids','predicted_response'}
assert not (required - set(sample.keys()))
assert len(sample['predicted_track_ids']) == 20
assert sample['predicted_response'].strip()
print(f'sample response[0]: {sample["predicted_response"][:300]!r}')
# Check that LM cites a specific track (not confused by multi-candidate).
# Qualitative probe: does the response mention a specific track name?

stage = '/content/_stage_prediction'
shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, 'prediction.json'))
!cd {stage} && rm -f /content/prediction.zip && zip -q /content/prediction.zip prediction.json
!unzip -l /content/prediction.zip

In [ ]:
# 7a) Browser download.
from google.colab import files
files.download('/content/prediction.zip')

In [ ]:
# 7b) Drive backup.
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
dst = '/content/drive/MyDrive/recsys2026-predictions'
os.makedirs(dst, exist_ok=True)
shutil.copy('/content/prediction.zip', f'{dst}/{TID}__prediction.zip')
shutil.copy(f'music-crs-baselines/exp/inference/blindset_A/{TID}.json', dst)
!ls -lh {dst}

## After scoring, interpretation:

- **LLM ≈ 3.15 or higher**: coupling hypothesis validated. All retrieval experiments can now be retried with the decoupled prompt — expect them to show real retrieval lift without the LLM regression that masked signal before. This is the BIG unlock.
- **LLM ≈ 3.15 (no change)**: top-3 doesn't hurt (good) but the LM isn't using the extra candidates (neutral). Retrieval experiments can resume but no expected LLM lift on its own.
- **LLM < 2.9**: LM got confused by 3 candidates in stock prompt format. Need targeted prompt change ('Choose the best match from the candidates below') to guide the LM. Revert to 021 until that's tested.